# Talk to Your Database — Colab UI (Qwen3 + QLoRA)

Runtime: Colab **Pro** GPU (L4 or comparable VRAM). Run cells **0 → 5** in order.

Procedure: clone the UI branch → point `ADAPTER_DIR` at the QLoRA folder →
write `ui_config.json` → start Streamlit → open the Colab proxy URL.

The base model is downloaded from Hugging Face. Only the adapter directory is supplied locally.
PEFT loads `adapter_config.json` and `adapter_model.safetensors` from that directory.
Sidebar backend should read `qwen3-4b+adapter`.


## Cell 0 — Clone repository


In [ ]:
import os, shutil
from pathlib import Path

REPO_URL = "https://github.com/siddhant-192/Group-10-DS-and-AI-Lab-Project.git"
BRANCH = "milestone-6-ui"  # change to "main" after the UI merge

def find_project_root(base=Path("/content")) -> Path:
    hits = sorted(base.glob("**/app/app.py"))
    hits = [h for h in hits if "/.git/" not in str(h).replace("\\", "/")]
    if not hits:
        raise FileNotFoundError("Could not find app/app.py under /content")
    return hits[0].resolve().parents[1]

dest = Path("/content/repo")
if dest.exists():
    shutil.rmtree(dest)
get_ipython().system(f'git clone --depth 1 -b {BRANCH} "{REPO_URL}" /content/repo')

ROOT = find_project_root()
os.environ["PROJECT_ROOT"] = str(ROOT)
print("PROJECT_ROOT =", ROOT)

models_py = ROOT / "app" / "backend" / "models.py"
if not models_py.is_file():
    raise FileNotFoundError(
        f"Missing {models_py}. Branch {BRANCH!r} does not contain the Streamlit UI."
    )
print("models.py present")


## Cell 1 — Dependencies and GPU


In [ ]:
import os
from pathlib import Path

ROOT = Path(os.environ["PROJECT_ROOT"])
os.chdir(ROOT)
print("cwd", ROOT)

get_ipython().system("pip install -q -r app/scripts/colab-ui-requirements.txt")

import torch
assert torch.cuda.is_available(), "Select a GPU runtime (Runtime → Change runtime type)."
print("CUDA:", torch.cuda.get_device_name(0))
print("VRAM GB (approx):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


## Cell 2 — Adapter directory

Set `ADAPTER_DIR` to the folder that contains `adapter_config.json` and
`adapter_model.safetensors` (for example `final_checkpoint_375_adapter_upload/`
after unzipping the release archive).

Default behaviour: mount Google Drive, search for that zip or folder, unzip if needed.
Override with `MANUAL_ADAPTER_DIR` when the path is already known.


In [ ]:
import os
import shutil
from pathlib import Path

USE_GOOGLE_DRIVE = True
ZIP_NAME_HINT = "final_checkpoint_375_adapter_upload"
EXTRACT_TO = Path("/content/final_checkpoint_375_adapter_upload")

# Example override:
# MANUAL_ADAPTER_DIR = Path("/content/drive/MyDrive/path/final_checkpoint_375_adapter_upload")
MANUAL_ADAPTER_DIR = None

def find_adapter_dir(start: Path) -> Path | None:
    hits = sorted(start.glob("**/adapter_config.json"))
    return hits[0].parent if hits else None

if MANUAL_ADAPTER_DIR is not None:
    ADAPTER_DIR = Path(MANUAL_ADAPTER_DIR)
else:
    if USE_GOOGLE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        search_root = Path("/content/drive/MyDrive")
    else:
        search_root = Path("/content")

    ADAPTER_DIR = find_adapter_dir(search_root)
    if ADAPTER_DIR is None:
        zips = [z for z in sorted(search_root.rglob("*.zip")) if ZIP_NAME_HINT in z.name]
        if not zips:
            raise FileNotFoundError(
                f"No zip matching {ZIP_NAME_HINT!r} under {search_root}. "
                "Place the adapter zip on Drive or set MANUAL_ADAPTER_DIR."
            )
        zip_path = zips[0]
        print("zip:", zip_path)
        get_ipython().system(f'unzip -q -o "{zip_path}" -d /content/')
        ADAPTER_DIR = find_adapter_dir(Path("/content"))
        if ADAPTER_DIR is None:
            raise FileNotFoundError("Unzip completed but adapter_config.json was not found.")

print("ADAPTER_DIR:", ADAPTER_DIR.resolve())
cfg = ADAPTER_DIR / "adapter_config.json"
weights = list(ADAPTER_DIR.glob("adapter_model*.safetensors")) + list(
    ADAPTER_DIR.glob("adapter_model*.bin")
)
print("adapter_config.json:", cfg.is_file())
print("weights:", [p.name for p in weights[:10]])
if not cfg.is_file():
    raise FileNotFoundError(f"Missing adapter_config.json under {ADAPTER_DIR}")

os.environ["ADAPTER_DIR"] = str(ADAPTER_DIR.resolve())


## Cell 3 — Write `ui_config.json`

Copies the Qwen3 example config and sets `adapter_dir` to `ADAPTER_DIR`.


In [ ]:
import json
import os
import shutil
from pathlib import Path

ROOT = Path(os.environ["PROJECT_ROOT"])
ADAPTER_DIR = Path(os.environ["ADAPTER_DIR"])
os.chdir(ROOT)

src = ROOT / "app" / "ui_config.qwen3.example.json"
dst = ROOT / "app" / "ui_config.json"
shutil.copy2(src, dst)

cfg = json.loads(dst.read_text(encoding="utf-8"))
cfg["backend"] = "qwen3-4b+adapter"
cfg["model_slug"] = "qwen3-4b-instruct-2507"
cfg["adapter_dir"] = str(ADAPTER_DIR)
cfg["load_4bit"] = True
cfg["max_new_tokens"] = 512
dst.write_text(json.dumps(cfg, indent=2), encoding="utf-8")

print(dst)
print(json.dumps(cfg, indent=2))


## Cell 4 — Demo databases and Streamlit

First load may take several minutes (base model download + adapter attach).


In [ ]:
import json
import os
import socket
import time
from pathlib import Path

ROOT = Path(os.environ["PROJECT_ROOT"])
os.chdir(ROOT)

get_ipython().system("python app/scripts/download_demo_databases.py")

cfg = json.loads((ROOT / "app" / "ui_config.json").read_text(encoding="utf-8"))
assert cfg.get("backend") == "qwen3-4b+adapter", cfg
assert cfg.get("adapter_dir"), "adapter_dir missing — re-run Cells 2–3"
print("backend:", cfg["backend"])
print("adapter_dir:", cfg["adapter_dir"])

cfg_dir = ROOT / ".streamlit"
cfg_dir.mkdir(exist_ok=True)
(cfg_dir / "config.toml").write_text(
    "\n".join([
        "[server]",
        "headless = true",
        "enableCORS = false",
        "enableXsrfProtection = false",
        "port = 8501",
        'address = "0.0.0.0"',
        "",
        "[browser]",
        "gatherUsageStats = false",
        "",
    ]),
    encoding="utf-8",
)

os.system("fuser -k 8501/tcp >/dev/null 2>&1")
time.sleep(1)

adapter = cfg["adapter_dir"]
cmd = (
    f"cd {ROOT} && "
    f"MODEL_BACKEND=qwen3-4b+adapter MODEL_SLUG=qwen3-4b-instruct-2507 "
    f"ADAPTER_DIR={adapter} "
    "nohup python -m streamlit run app/app.py "
    "--server.port 8501 --server.address 0.0.0.0 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false "
    "--browser.gatherUsageStats false "
    "> /tmp/streamlit_ui.log 2>&1 &"
)
get_ipython().system_raw(cmd)
print("Streamlit starting…")

def port_open(port: int = 8501) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True
    except OSError:
        return False

ok = False
for i in range(90):
    if port_open():
        ok = True
        break
    time.sleep(1)
    if i % 15 == 14:
        print(f"  waiting… {i+1}s")

if not ok:
    print("Port 8501 did not open. Log:")
    get_ipython().system("tail -n 80 /tmp/streamlit_ui.log")
else:
    print("Port 8501 is open. Continue with Cell 5.")


## Cell 5 — Colab proxy URL


In [ ]:
import socket
import time
from IPython.display import display, HTML
from google.colab.output import eval_js

def port_open(port: int = 8501) -> bool:
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True
    except OSError:
        return False

if not port_open():
    raise RuntimeError("Port 8501 is closed. Re-run Cell 4, then this cell.")

time.sleep(1)
url = eval_js("google.colab.kernel.proxyPort(8501)")
print("backend: qwen3-4b+adapter")
print(url)
display(HTML(f'<p><a href="{url}" target="_blank">Open UI</a></p>'))


## Optional — stop Streamlit


In [ ]:
import os
os.system("fuser -k 8501/tcp >/dev/null 2>&1")
print("Stopped listeners on port 8501.")
